# Phase 0.5 · Preview the decision prompts

Print the yes/no pathogen-evidence prompt and one feature prompt resolved for a pathogen–host pair in the authored V4 run. The pair feature is selected from the host-aware profile, so its question and context reflect the chosen pathogen group and host. This notebook makes no model or network calls. Paste a title and abstract in the controls to inspect the fully populated prompt; otherwise explicit placeholders are printed.

## Step 1 · Load the run and prompt/profile definitions

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
from pathogen_atlas_v4.features import resolve_profile
from pathogen_atlas_v4.prompts import build_pathogen_screen_prompt, build_pair_feature_prompt
from pathogen_atlas_v4.runs import resolve_run

ROOT = PROJECT_ROOT
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('01_prompt_preview', RUN_ID)
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
PATHOGENS = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
CATALOGUE = yaml.safe_load((ROOT / CONFIG['features']['catalogue']).read_text(encoding='utf-8'))
PROMPT_PATH = ROOT / CONFIG.get('decisions', {}).get('prompt_catalogue', 'configs/prompts.yaml')
PROMPTS = yaml.safe_load(PROMPT_PATH.read_text(encoding='utf-8'))
print('Run:', RUN_ID, '| taxonomy:', TAXONOMY_ROOT.name, '| prompt set:', PROMPTS['prompt_set_version'])
print('Log:', LOG_PATH)
LOG.info('Loaded prompt preview inputs: run=%s taxonomy=%s prompt_set=%s', RUN_ID, TAXONOMY_ROOT.name, PROMPTS['prompt_set_version'])

## Step 2 · Resolve one pathogen–host pair to preview

In [ ]:
# Leave either ID as None to use the first resolved selection. Set an explicit ID to inspect a different member of the run.
PREVIEW_PATHOGEN_ID = None
PREVIEW_HOST_ID = None
PREVIEW_ARTICLE_TITLE = ''
PREVIEW_ARTICLE_ABSTRACT = ''
PREVIEW_FEATURE_ID = None  # None selects the first applicable pathogen_host_pair feature.

host_groups = {key: group.host_id.astype(str).tolist() for key, group in HOSTS.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in PATHOGENS.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)),
    known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)),
    host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
if not resolved.pathogen_ids or not resolved.host_ids:
    raise ValueError('The run must resolve at least one pathogen and one host before prompt preview.')
pathogen_id = str(PREVIEW_PATHOGEN_ID or resolved.pathogen_ids[0])
host_id = str(PREVIEW_HOST_ID or resolved.host_ids[0])
if pathogen_id not in set(resolved.pathogen_ids):
    raise ValueError(f'{pathogen_id} is not in the resolved run pathogen selection.')
if host_id not in set(resolved.host_ids):
    raise ValueError(f'{host_id} is not in the resolved run host selection.')
pathogen = PATHOGENS.set_index('pathogen_id').loc[pathogen_id]
host = HOSTS.set_index('host_id').loc[host_id]
print(f"Pathogen: {pathogen.accepted_name} [{pathogen.pathogen_group}] ({pathogen_id})")
print(f"Host: {host.accepted_name} [{host.host_group}] ({host_id})")
LOG.info('Prompt preview pair selected: pathogen=%s host=%s', pathogen_id, host_id)

## Step 3 · Print the pathogen yes/no screening query

In [ ]:
pathogen_prompt = build_pathogen_screen_prompt(
    pathogen_name=str(pathogen.accepted_name), pathogen_id=pathogen_id,
    article_title=PREVIEW_ARTICLE_TITLE, article_abstract=PREVIEW_ARTICLE_ABSTRACT,
    prompt_config=PROMPTS)
print(json.dumps(pathogen_prompt, indent=2, ensure_ascii=False))
display(Markdown(f"**Question version:** `{pathogen_prompt['question_version']}` · **Prompt version:** `{pathogen_prompt['prompt_version']}`"))
LOG.info('Printed pathogen screening prompt question=%s prompt=%s', pathogen_prompt['question_version'], pathogen_prompt['prompt_version'])

## Step 4 · Print one pair-specific feature query

In [ ]:
profile = resolve_profile(CATALOGUE,
    pathogen_group=str(pathogen.pathogen_group), host_group=str(host.host_group),
    host_taxon_id=str(host.taxon_id), relationship_context='disease',
    profile_version=str(CONFIG['features']['profile_version']))
pair_features = [feature for feature in profile['ordered_features'] if feature.get('scope') == 'pathogen_host_pair']
if PREVIEW_FEATURE_ID:
    pair_features = [feature for feature in pair_features if feature['feature_id'] == PREVIEW_FEATURE_ID]
if not pair_features:
    raise ValueError('No selected pathogen_host_pair feature is available in this resolved profile.')
feature = pair_features[0]
feature_prompt = build_pair_feature_prompt(
    pathogen_name=str(pathogen.accepted_name), pathogen_id=pathogen_id,
    host_name=str(host.accepted_name), host_id=host_id, feature=feature,
    article_title=PREVIEW_ARTICLE_TITLE, article_abstract=PREVIEW_ARTICLE_ABSTRACT,
    prompt_config=PROMPTS)
print('Resolved profile:', profile['profile_id'], profile['profile_version'], profile['resolved_profile_hash'])
print(json.dumps(feature_prompt, indent=2, ensure_ascii=False))
LOG.info('Printed pair feature prompt: feature=%s question=%s prompt=%s profile=%s',
         feature['feature_id'], feature.get('question_version'), feature_prompt['prompt_version'], profile['resolved_profile_hash'])

## Review notes

These are the exact versioned message payloads to hand to a provider adapter, with the run-selected pathogen and host names inserted. To preview against a real example, enter a PubMed title and abstract in Step 2 and rerun Steps 2–4. No decision is made or cached by this preview.